In [1]:
# %%
# ============================================================
# STEP 1: PV ECONOMIC POSITIVITY ANALYSIS (OVERVIEW + SETUP)
# Description (Markdown):
"""
# Photovoltaic Economic Positivity Statistical Analysis (No statsmodels version)

This script analyses whether photovoltaic (PV) retrofit options are
economically positive (binary 0/1) across a set of rural Scottish dwellings.

## Target variables:
- PV_ECON_POS_FIXED
- PV_ECON_POS_FLEX

## Methods used:
1. Chi-square test of independence
2. Fisher’s Exact Test (for 2x2 tables)
3. Cramér’s V (effect size)
4. Logistic regression using scikit-learn (odds ratios approximated from coefficients)

⚠️ Note:
- p-values for logistic regression are NOT computed (sklearn limitation)
- Odds ratios are derived from model coefficients

## Grouping variables tested:
- EPC_GROUP
- AGE_GROUP
- FUEL_GROUP
- HEATING_GROUP
- BUILT_FORM_GROUP

Outputs:
- Statistical results CSV per HP scenario (fixed + flex)
- Saved to user-defined project directories
"""
# ============================================================

import os
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.linear_model import LogisticRegression

# ----------------------------
# Load dataset
# ----------------------------
file_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_D2_and_SD2_RESULTS.csv"
df = pd.read_csv(file_path)
df.columns = df.columns.str.strip()

# ----------------------------
# Targets
# ----------------------------
target_fixed = "PV_ECON_POS_FIXED"
target_flex = "PV_ECON_POS_FLEX"

df = df.dropna(subset=[target_fixed, target_flex], how="all")

print(f"Dataset loaded: {len(df)} rows")

Dataset loaded: 54 rows


In [2]:
# %%
# ============================================================
# STEP 2: CREATE OUTPUT DIRECTORIES + GROUPING VARIABLES
# ============================================================

fig_dir = "/Users/jakegehrung/Desktop/data_raw/RESULTS_FIGURES/HP_RESULTS"
os.makedirs(fig_dir, exist_ok=True)

# ----------------------------
# EPC grouping
# ----------------------------
def group_epc(rating):
    if pd.isna(rating):
        return np.nan
    rating = str(rating).upper()
    if rating in ["A", "B"]:
        return "A_to_B"
    elif rating in ["C", "D"]:
        return "C_to_D"
    elif rating in ["E", "F", "G"]:
        return "E_to_G"
    return np.nan

df["EPC_GROUP"] = df["CURRENT_ENERGY_RATING"].apply(group_epc)

# ----------------------------
# Age grouping
# ----------------------------
def group_age(age):
    if pd.isna(age):
        return np.nan
    age = str(age).lower()

    if "before 1919" in age or "1930-1949" in age:
        return "Before 1919 to 1949"
    elif "1950-1964" in age or "1965-1975" in age:
        return "1950-1975"
    elif "1976-1983" in age or "1984-1991" in age:
        return "1976-1991"
    elif any(x in age for x in ["1992-1998", "1999-2002", "2003-2007", "2008"]):
        return "1992 onwards"
    return np.nan

df["AGE_GROUP"] = df["CONSTRUCTION_AGE_BAND"].apply(group_age)

# ----------------------------
# Fuel grouping
# ----------------------------
def group_fuel(fuel):
    if pd.isna(fuel):
        return np.nan
    fuel = str(fuel).lower()
    return "Electric" if "electric" in fuel else "Other Fuel"

df["FUEL_GROUP"] = df["MAIN_FUEL"].apply(group_fuel)

# ----------------------------
# Heating grouping
# ----------------------------
def group_heating(system):
    if pd.isna(system):
        return np.nan
    return "Heat Pump" if "heat pump" in str(system).lower() else "Other Heating System"

df["HEATING_GROUP"] = df["MAIN_HEATING_CATEGORY"].apply(group_heating)

df["BUILT_FORM_GROUP"] = df["BUILT_FORM"]

group_cols = ["EPC_GROUP", "AGE_GROUP", "FUEL_GROUP", "HEATING_GROUP", "BUILT_FORM_GROUP"]

print("Grouping complete.")

Grouping complete.


In [3]:
# %%
# ============================================================
# STEP 3: STATISTICAL TEST FUNCTIONS (NO statsmodels)
# ============================================================

def cramers_v(confusion_matrix):
    chi2 = stats.chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    r, k = confusion_matrix.shape
    return np.sqrt((chi2 / n) / (min(r - 1, k - 1) + 1e-12))


def chi_square_or_fisher(df, group_col, target_col):
    table = pd.crosstab(df[group_col], df[target_col])

    if table.shape == (2, 2):
        oddsratio, p = stats.fisher_exact(table)
        test_name = "Fisher Exact"
        stat = oddsratio
        effect = np.nan
    else:
        chi2, p, _, _ = stats.chi2_contingency(table)
        stat = chi2
        effect = cramers_v(table)
        test_name = "Chi-square"

    return test_name, stat, p, effect, table


def logistic_regression_sklearn(df, group_col, target_col):
    d = df[[group_col, target_col]].dropna()

    X = pd.get_dummies(d[group_col], drop_first=True)
    y = d[target_col]

    if X.shape[1] == 0:
        return None, None

    model = LogisticRegression(max_iter=1000)
    model.fit(X, y)

    odds_ratios = dict(zip(X.columns, np.exp(model.coef_[0])))

    return {"intercept": np.exp(model.intercept_[0]), **odds_ratios}

In [4]:
# %%
# ============================================================
# STEP 4: RUN ANALYSIS (HP FIXED)
# ============================================================

results_fixed = []
df_fixed = df.dropna(subset=[target_fixed])

for col in group_cols:
    try:
        test_name, stat, p, effect, table = chi_square_or_fisher(df_fixed, col, target_fixed)
        odds = logistic_regression_sklearn(df_fixed, col, target_fixed)

        results_fixed.append({
            "Scenario": "FIXED",
            "Grouping": col,
            "Test": test_name,
            "Statistic": stat,
            "p-value": p,
            "Effect Size (CramersV if applicable)": effect,
            "Logistic Odds Ratios (sklearn)": str(odds),
            "Contingency Table": str(table.to_dict())
        })

        print(f"Fixed PV complete: {col}")

    except Exception as e:
        print(f"Error (FIXED, {col}): {e}")


fixed_out = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_PV_FIXED_STATS.csv"
pd.DataFrame(results_fixed).to_csv(fixed_out, index=False)

print(f"Saved FIXED results → {fixed_out}")

Fixed PV complete: EPC_GROUP
Fixed PV complete: AGE_GROUP
Fixed PV complete: FUEL_GROUP
Fixed PV complete: HEATING_GROUP
Fixed PV complete: BUILT_FORM_GROUP
Saved FIXED results → /Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_PV_FIXED_STATS.csv


In [5]:
# %%
# ============================================================
# STEP 5: RUN ANALYSIS (HP FLEX)
# ============================================================

results_flex = []
df_flex = df.dropna(subset=[target_flex])

for col in group_cols:
    try:
        test_name, stat, p, effect, table = chi_square_or_fisher(df_flex, col, target_flex)
        odds = logistic_regression_sklearn(df_flex, col, target_flex)

        results_flex.append({
            "Scenario": "FLEX",
            "Grouping": col,
            "Test": test_name,
            "Statistic": stat,
            "p-value": p,
            "Effect Size (CramersV if applicable)": effect,
            "Logistic Odds Ratios (sklearn)": str(odds),
            "Contingency Table": str(table.to_dict())
        })

        print(f"Flex PV complete: {col}")

    except Exception as e:
        print(f"Error (FLEX, {col}): {e}")


flex_out = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_PV_FLEX_STATS.csv"
pd.DataFrame(results_flex).to_csv(flex_out, index=False)

print(f"Saved FLEX results → {flex_out}")

Flex PV complete: EPC_GROUP
Flex PV complete: AGE_GROUP
Flex PV complete: FUEL_GROUP
Flex PV complete: HEATING_GROUP
Flex PV complete: BUILT_FORM_GROUP
Saved FLEX results → /Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_PV_FLEX_STATS.csv
